In [ ]:
# CCX WP24 shard 00 — CONFIG (Plan Sec 8)
import os, json
TAG='wp24'
SHARD_ID=0
OUT='/content/ccx_wp24_shard00.csv'
CONFIG_JSON='{"note": "full pain map, single shard"}'
CONFIG=json.loads(CONFIG_JSON)
print(TAG, 'shard', SHARD_ID, 'groups', len(CONFIG.get('groups', CONFIG.get('jobs', []))))


In [ ]:
# ---- setup: clone pinned repo + deps (thin) ----
import os, sys, subprocess, pathlib, json, time, hashlib

REPO_DIR = "/tmp/ccx"
GIT_URL = "https://github.com/hugogobato/ccx-contextual-confounding.git"
GIT_SHA = "d2dd265cef5ef171684997e8115126d75a2dd4cd"

# pip deps (numpy/scipy/pandas/matplotlib already on Colab, ensure versions)
# keep install light; inflation not needed for these jobs but harmless
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy==1.17.1", "pandas==3.0.1"])
except Exception as e:
    print("pip install warning:", e)

# clone or reuse
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"cloning {GIT_URL} @ {GIT_SHA[:7]} -> {REPO_DIR}")
    # try anonymous clone first (works if repo public); if private, try token from Colab secrets
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get("CCX_GH_TOKEN")
    except Exception:
        tok = os.environ.get("CCX_GH_TOKEN")
    url = GIT_URL
    if tok:
        # inject token: https://<token>@github.com/...
        url = GIT_URL.replace("https://", f"https://{tok}@")
        print("using GH token from secrets/env")
    subprocess.check_call(["git", "clone", url, REPO_DIR])
else:
    print("repo already cloned, fetching")
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])

subprocess.check_call(["git", "-C", REPO_DIR, "checkout", GIT_SHA, "-q"])
# ensure src on path and ROOT env for imports
if REPO_DIR + "/src" not in sys.path:
    sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)
print("checked out", subprocess.check_output(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"], text=True).strip())
# verify
import numpy, scipy, pandas
print("numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__)
CODE_HASH = GIT_SHA[:16]


In [ ]:

# ---- WP2.4 scaling pain map driver (single shard) ----
import os, json, time, pandas as pd
from run_wp24_scaling import bench_cell, get_battery_cached, build_iv_A_general
from pathlib import Path
import numpy as np
# bench_cell already does per-cell loops; we just call main-like logic but write to OUT

# reuse bench logic but ensure OUT path is shard file
from run_wp24_scaling import ROOT as WP24_ROOT
import run_wp24_scaling as wp24mod

# monkey-patch RES to point to /content for this shard? Instead run inline bench and write to OUT
# Copy bench loop from run_wp24_scaling.main but write to OUT

import json as _json
from pathlib import Path as _P
ROOT_SHARD = Path("/tmp/ccx")
# run bench
cfg=json.loads((ROOT_SHARD / "configs" / "seeds.json").read_text())["phase2"]
rows=[]
for kz,kx,ky in [tuple(c) for c in cfg["alphabet_cells"]]:
    for n in (500,2000,8000):
        t0=time.time()
        rows+=bench_cell(kz,kx,ky,n)
        print(f"[wp24] ({kz},{kx},{ky}) n={n}: {time.time()-t0:.1f}s", flush=True)
df=pd.DataFrame(rows)
# sweep
sweep=[]
for (kx,ky) in [(2,2),(2,5),(3,5),(2,8)]:
    for kz in (2,3,4):
        A=build_iv_A_general(kz,kx,ky)
        Mint=A.astype(int)
        K,Q=A.shape
        for r in range(3):
            rng=np.random.default_rng(77000+r)
            cond=np.concatenate([rng.dirichlet(np.ones(K//kz)) for _ in range(kz)])
            from run_wp24_scaling import timed
            from witness_estimators import cf1_plugin_stat, slack_plugin_stat
            dt1,m1,_=timed(cf1_plugin_stat, Mint, np.round(cond*4000), kz)
            dt2,m2,_=timed(slack_plugin_stat, Mint, np.round(cond*4000), kz)
            sweep.append({"kz":kz,"kx":kx,"ky":ky,"K":K,"Q":Q,"method":"cf1_lp","seconds":dt1,"peak_bytes":m1})
            sweep.append({"kz":kz,"kx":kx,"ky":ky,"K":K,"Q":Q,"method":"slack_lp","seconds":dt2,"peak_bytes":m2})
if sweep:
    df=pd.concat([df, pd.DataFrame(sweep)], ignore_index=True)
df.to_csv(OUT, index=False)
print(f"wrote {len(df)} rows -> {OUT}")

manifest={"tag": TAG, "shard_id": SHARD_ID, "code_hash": CODE_HASH, "rows": len(df), "git_sha": GIT_SHA}
mpath=f"/content/ccx_{TAG}_manifest_shard{SHARD_ID:02d}.json"
with open(mpath,"w") as fh: json.dump(manifest,fh,indent=2)
print("MANIFEST", json.dumps(manifest))
try:
    from google.colab import files
    files.download(OUT); files.download(mpath)
except Exception as e:
    print("(dl skip)", e)
